In [1]:
import os
import sys
import torch
import importlib
import pandas as pd
from sklearn.linear_model import Ridge

import utils
import evaluation_metrics as em
from datasets import load_dataset
from hkan.hkan import (
    Sigmoid, Gaussian, ReLU, Tanh, Softplus, Identity,
    make_hkan_layer, extend_hkan, set_tqdm_disable,
)
importlib.reload(utils)

/Users/gizemnurdal/miniconda3/envs/swim-meets-kans/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'utils' from '/Users/gizemnurdal/Workspace/swim-meets-kans/utils.py'>

# TF1 Experiments

## HKAN

In [2]:
# Load TF1 dataset
tf1 = load_dataset("TF1")
tf1_x_train, tf1_x_test, tf1_y_train, tf1_y_test = tf1["train_input"], tf1["test_input"], tf1["train_label"], tf1["test_label"]

In [3]:
print("TF1 train describe")
display(pd.DataFrame(tf1_x_train).describe())
print("TF1 test describe")
display(pd.DataFrame(tf1_x_test).describe())
print("TF1 y_train describe")
display(pd.Series(tf1_y_train).describe())
print("TF1 y_test describe")
display(pd.Series(tf1_y_test).describe())

TF1 train describe


,0,1
count,5000.000000,5000.000000
mean,0.502492,0.498871
std,0.291808,0.285606
min,0.000021,0.000020
25%,0.247467,0.253435
50%,0.504131,0.497807
75%,0.758499,0.743126
max,0.999705,0.999692


TF1 test describe


,0,1
count,10000.000000,10000.000000
mean,0.500000,0.500000
std,0.291591,0.291591
min,0.000000,0.000000
25%,0.250000,0.250000
50%,0.500000,0.500000
75%,0.750000,0.750000
max,1.000000,1.000000


TF1 y_train describe


count    5000.000000
mean        0.505381
std         0.166339
min         0.009263
25%         0.414095
50%         0.501150
75%         0.600623
max         0.988818
dtype: float64

TF1 y_test describe


count    10000.000000
mean         0.500000
std          0.170042
min          0.000000
25%          0.404882
50%          0.500000
75%          0.595118
max          1.000000
dtype: float64

In [4]:
# TF1 HKAN model params
tf1_model_params = [
    {
        "layer": 0,
        "n_vars_out": 932,
        "basis_fn": Sigmoid(s=1),
        "n_basis": 2,
        "centers": "random_data_points",
        "expanding_base_regressor": Ridge(alpha=0.1),
    },
    {
        "layer": 1,
        "n_vars_out": 1,
        "basis_fn": Tanh(s=33),
        "n_basis": 13,
        "centers": "random_data_points",
        "expanding_base_regressor": Ridge(alpha=10),
    }
]
tf1_model = utils.build_hkan_model_from_configs(tf1_model_params)
tf1_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('expanding_layer_0', ...), ('connecting_layer_0', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_vars_out,932
,n_basis,2
,centers,'random_data_points'
,basis_fn,<hkan.hkan.Si...t 0x15bff6b90>
,base_regressor,Ridge(alpha=0.1)
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True


In [5]:
# Fit HKAN for TF1
utils.fit_hkan(tf1_model,  tf1_x_train, tf1_y_train)
tf1_results = em.evaluate(
    tf1_model,
    tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test
)

Fitting connecting regressors: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]



✓ HKAN training completed in 4.4770 seconds


In [6]:
em.print_results(tf1_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             1.468331e-11         1.683097e-11        
RMSE            1.971031e-11         3.007025e-11        
-------------------------------------------------------
Inference (s)   1.305806             2.078614            


## KAN

In [7]:
tf1_kan_study = utils.study_optuna_kan("TF1", tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test, n_trials = 10)

Results found for TF1. Loading from data/tf1_kan_optuna_search.csv
Best CV Validation RMSE: 0.161730
Best Test RMSE:          0.172035
Best width_idx:          2
Best architecture:       [2, 3, 1]

All 10 trials:
   number     value  params_width_idx  val_rmse  test_rmse
0       0  0.161730                 2  0.161730   0.172035
1       1  0.568877                 8  0.568877   0.274214
2       2  0.168665                 4  0.168665   0.298363
3       3  3.752602                 9  3.752602   0.568879
4       4  0.174820                 1  0.174820   0.179805
5       5  0.245464                 6  0.245464   0.178054
6       6  0.198778                 7  0.198778   0.172921
7       7  0.198986                 3  0.198986   0.190309
8       8  0.166398                 0  0.166398   0.170176
9       9  0.573112                 5  0.573112   1.370827


In [8]:
# Create TF1 KAN model with best architecture
tf1_kan_model = utils.build_kan(width=tf1_kan_study["architecture"], grid=3, k=3, seed=42)

checkpoint directory created: ./model
saving model version 0.0


In [9]:
# Train and test TF1 KAN model
utils.fit_kan(tf1_kan_model, tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test, steps=100, opt="Adam", lamb=1e-3)

tf1_kan_results = em.evaluate(
    utils.KANModel(tf1_kan_model),
    tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test
)

| train_loss: 1.67e-01 | test_loss: 1.71e-01 | reg: 2.88e+01 | : 100%|█| 100/100 [00:22<00:00,  4.43


saving model version 0.1

✓ KAN training completed in 22.5897 seconds


In [10]:
em.print_results(tf1_kan_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             1.261283e-01         1.289797e-01        
RMSE            1.681006e-01         1.720347e-01        
-------------------------------------------------------
Inference (s)   0.010028             0.009496            


# TF2 Experiments

## HKAN

In [11]:
# Load TF2 dataset
tf2 = load_dataset("TF2")
tf2_x_train, tf2_x_test, tf2_y_train, tf2_y_test = tf2["train_input"], tf2["test_input"], tf2["train_label"], tf2["test_label"]

In [12]:
print("TF2 train describe")
display(pd.DataFrame(tf2_x_train).describe())
print("TF2 test describe")
display(pd.DataFrame(tf2_x_test).describe())
print("TF2 y_train describe")
display(pd.Series(tf2_y_train).describe())
print("TF2 y_test describe")
display(pd.Series(tf2_y_test).describe())

TF2 train describe


,0,1
count,5000.000000,5000.000000
mean,0.502861,0.502000
std,0.285424,0.285515
min,0.000133,0.000387
25%,0.259877,0.257710
50%,0.504340,0.502470
75%,0.747827,0.747088
max,0.999850,0.999980


TF2 test describe


,0,1
count,10000.000000,10000.000000
mean,0.500000,0.500000
std,0.291591,0.291591
min,0.000000,0.000000
25%,0.250002,0.250002
50%,0.500000,0.500000
75%,0.749997,0.749997
max,1.000000,1.000000


TF2 y_train describe


count    5000.000000
mean        0.482074
std         0.173621
min        -0.103030
25%         0.356747
50%         0.477600
75%         0.603375
max         1.145400
dtype: float64

TF2 y_test describe


count    10000.000000
mean         0.481887
std          0.129811
min          0.000000
25%          0.405035
50%          0.478680
75%          0.555480
max          1.000000
dtype: float64

In [13]:
# TF2 HKAN model params
tf2_model_params = [
    {
        "layer": 0,
        "n_vars_out": 48,
        "basis_fn": Tanh(s=22),
        "n_basis": 17,
        "centers": "random",
        "expanding_base_regressor": Ridge(alpha=0.1),
    },
    {
        "layer": 1,
        "n_vars_out": 11,
        "basis_fn": Softplus(s=18),
        "n_basis": 21,
        "centers": "random_data_points",
        "expanding_base_regressor": Ridge(alpha=1),
    },
    {
        "layer": 2,
        "n_vars_out": 1,
        "basis_fn": Identity(),
        # Default configuration
        "n_basis": 1,
        "centers": "equally_spaced",
        "expanding_base_regressor": None,
    }
]
tf2_model = utils.build_hkan_model_from_configs(tf2_model_params)
tf2_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('expanding_layer_0', ...), ('connecting_layer_0', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_vars_out,48
,n_basis,17
,centers,'random'
,basis_fn,<hkan.hkan.Ta...t 0x15c034890>
,base_regressor,Ridge(alpha=0.1)
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True


In [14]:
# Fit HKAN for TF2
utils.fit_hkan(tf2_model, tf2_x_train, tf2_y_train)
tf2_results = em.evaluate(
    tf2_model,
    tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test
)

Fitting connecting regressors: 100%|██████████| 1/1 [00:00<00:00, 447.06it/s]



✓ HKAN training completed in 2.3933 seconds


In [15]:
em.print_results(tf2_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             1.008521e-01         1.348625e-02        
RMSE            1.163214e-01         1.698008e-02        
-------------------------------------------------------
Inference (s)   0.981312             1.931742            


## KAN

In [16]:
tf2_kan_study = utils.study_optuna_kan("TF2", tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test, n_trials=10)

Results found for TF2. Loading from data/tf2_kan_optuna_search.csv
Best CV Validation RMSE: 0.173447
Best Test RMSE:          0.129826
Best width_idx:          0
Best architecture:       [2, 1]

All 10 trials:
   number     value  params_width_idx  val_rmse  test_rmse
0       0  0.220154                 2  0.220154   0.134638
1       1  0.255157                 8  0.255157  10.841902
2       2  0.214038                 4  0.214038   0.239839
3       3  2.014279                 9  2.014279   0.431603
4       4  0.242890                 1  0.242890   0.132069
5       5  0.198356                 6  0.198356   0.130486
6       6  0.313805                 7  0.313805   0.488357
7       7  0.197938                 3  0.197938   0.181680
8       8  0.173447                 0  0.173447   0.129826
9       9  4.529024                 5  4.529024   0.473081


In [17]:
tf2_kan_model = utils.build_kan(width=tf2_kan_study["architecture"], grid=3, k=3, seed=42)

checkpoint directory created: ./model
saving model version 0.0


In [18]:
utils.fit_kan(tf2_kan_model, tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test, steps=100, opt="Adam", lamb=1e-3)

tf2_kan_results = em.evaluate(
    utils.KANModel(tf2_kan_model),
    tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test
)

| train_loss: 1.74e-01 | test_loss: 1.30e-01 | reg: 3.84e-01 | : 100%|█| 100/100 [00:23<00:00,  4.20

saving model version 0.1

✓ KAN training completed in 23.8396 seconds


In [19]:
em.print_results(tf2_kan_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             1.403017e-01         9.761542e-02        
RMSE            1.734280e-01         1.298259e-01        
-------------------------------------------------------
Inference (s)   0.002746             0.003762            


# TF3 Experiments

## HKAN

In [20]:
# Load TF3 dataset
tf3 = load_dataset("TF3")
tf3_x_train, tf3_x_test, tf3_y_train, tf3_y_test = tf3["train_input"], tf3["test_input"], tf3["train_label"], tf3["test_label"]

In [21]:
print("TF3 train describe")
display(pd.DataFrame(tf3_x_train).describe())
print("TF3 test describe")
display(pd.DataFrame(tf3_x_test).describe())
print("TF3 y_train describe")
display(pd.Series(tf3_y_train).describe())
print("TF3 y_test describe")
display(pd.Series(tf3_y_test).describe())

TF3 train describe


,0,1
count,5000.000000,5000.000000
mean,0.504802,0.500059
std,0.286061,0.284858
min,0.000433,0.000133
25%,0.261751,0.255484
50%,0.507392,0.498916
75%,0.749510,0.745363
max,0.999891,0.999982


TF3 test describe


,0,1
count,10000.000000,10000.000000
mean,0.500000,0.500000
std,0.291591,0.291591
min,0.000000,0.000000
25%,0.250000,0.250000
50%,0.500000,0.500000
75%,0.750000,0.750000
max,1.000000,1.000000


TF3 y_train describe


count    5000.000000
mean        0.502180
std         0.161040
min         0.005368
25%         0.392510
50%         0.501351
75%         0.612301
max         0.990860
dtype: float64

TF3 y_test describe


count    10000.000000
mean         0.500000
std          0.163565
min          0.000000
25%          0.388427
50%          0.500000
75%          0.611573
max          1.000000
dtype: float64

In [22]:
# TF3 HKAN model params
tf3_model_params = [
    {
        "layer": 0,
        "n_vars_out": 924,
        "basis_fn": Tanh(s=50),
        "n_basis": 39,
        "centers": "random",
        "expanding_base_regressor": Ridge(alpha=0.001),
    },
    {
        "layer": 1,
        "n_vars_out": 1,
        "basis_fn": Identity(),
        "n_basis": 1,
        "centers": "equally_spaced",
        "expanding_base_regressor": None,
    }
]
tf3_model = utils.build_hkan_model_from_configs(tf3_model_params)
tf3_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('expanding_layer_0', ...), ('connecting_layer_0', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_vars_out,924
,n_basis,39
,centers,'random'
,basis_fn,<hkan.hkan.Ta...t 0x30ec03690>
,base_regressor,Ridge(alpha=0.001)
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True


In [23]:
# Fit HKAN for TF3
utils.fit_hkan(tf3_model, tf3_x_train, tf3_y_train)
tf3_results = em.evaluate(
    tf3_model,
    tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test
)

Fitting connecting regressors: 100%|██████████| 1/1 [00:00<00:00,  4.51it/s]



✓ HKAN training completed in 13.4030 seconds


In [24]:
em.print_results(tf3_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             3.754842e-06         3.888684e-06        
RMSE            9.324210e-06         6.590567e-06        
-------------------------------------------------------
Inference (s)   4.208641             7.534916            


## KAN

In [25]:
tf3_kan_study = utils.study_optuna_kan("TF3", tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test, n_trials=10)

Results found for TF3. Loading from data/tf3_kan_optuna_search.csv
Best CV Validation RMSE: 0.162384
Best Test RMSE:          0.164722
Best width_idx:          0
Best architecture:       [2, 1]

All 10 trials:
   number     value  params_width_idx  val_rmse  test_rmse
0       0  0.399037                 2  0.399037   0.160471
1       1  0.172992                 8  0.172992   0.164513
2       2  0.286203                 4  0.286203   0.163476
3       3  0.999206                 9  0.999206   0.163668
4       4  0.164660                 1  0.164660   0.487747
5       5  0.163563                 6  0.163563   0.163949
6       6  1.370679                 7  1.370679   0.361969
7       7  0.170057                 3  0.170057   0.182046
8       8  0.162384                 0  0.162384   0.164722
9       9  0.222863                 5  0.222863   0.196609


In [26]:
tf3_kan_model = utils.build_kan(width=tf3_kan_study["architecture"], grid=3, k=3, seed=42)

checkpoint directory created: ./model
saving model version 0.0


In [27]:
utils.fit_kan(tf3_kan_model, tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test, steps=100, opt="Adam", lamb=1e-3)

tf3_kan_results = em.evaluate(
    utils.KANModel(tf3_kan_model),
    tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test
)

| train_loss: 1.61e-01 | test_loss: 1.64e-01 | reg: 2.67e-01 | : 100%|█| 100/100 [00:19<00:00,  5.20


saving model version 0.1

✓ KAN training completed in 19.2492 seconds


In [28]:
em.print_results(tf3_kan_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             1.298408e-01         1.316747e-01        
RMSE            1.622684e-01         1.647225e-01        
-------------------------------------------------------
Inference (s)   0.003200             0.004372            


# TF4 Experiments

## HKAN

In [29]:
# Load TF4 dataset
tf4 = load_dataset("TF4")
tf4_x_train, tf4_x_test, tf4_y_train, tf4_y_test = tf4["train_input"], tf4["test_input"], tf4["train_label"], tf4["test_label"]

In [30]:
print("TF4 train describe")
display(pd.DataFrame(tf4_x_train).describe())
print("TF4 test describe")
display(pd.DataFrame(tf4_x_test).describe())
print("TF4 y_train describe")
display(pd.Series(tf4_y_train).describe())
print("TF4 y_test describe")
display(pd.Series(tf4_y_test).describe())

TF4 train describe


,0,1,2,3,4,5,6,7,8,9
count,3750.000000,3750.000000,3750.000000,3750.000000,3750.000000,3750.000000,3750.000000,3750.000000,3750.000000,3750.000000
mean,0.506746,0.503204,0.498323,0.500772,0.497799,0.496639,0.495851,0.498425,0.493649,0.496010
std,0.283847,0.285025,0.293268,0.293257,0.288584,0.290085,0.288278,0.291463,0.286498,0.289600
min,0.000000,0.000254,0.000092,0.000000,0.000000,0.000227,0.000000,0.000000,0.000000,0.000000
25%,0.265375,0.260039,0.239920,0.244900,0.244162,0.242192,0.243826,0.241701,0.244787,0.240751
50%,0.510035,0.501595,0.495827,0.498848,0.498117,0.495226,0.487462,0.506830,0.498333,0.495502
75%,0.750690,0.744755,0.755315,0.756410,0.748974,0.751233,0.741759,0.750584,0.740535,0.749713
max,1.000000,0.999900,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


TF4 test describe


,0,1,2,3,4,5,6,7,8,9
count,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000
mean,0.498333,0.490395,0.486467,0.497279,0.513013,0.496188,0.497025,0.501470,0.506500,0.495865
std,0.293236,0.284424,0.290277,0.276920,0.288045,0.293981,0.289387,0.297975,0.287566,0.291880
min,0.000048,0.000000,0.000000,0.000001,0.000004,0.000000,0.000444,0.000265,0.000719,0.000455
25%,0.246033,0.245511,0.232735,0.259110,0.270409,0.248973,0.251188,0.239863,0.261810,0.242328
50%,0.496355,0.488314,0.476912,0.494997,0.524857,0.495540,0.494254,0.500964,0.511718,0.484211
75%,0.745779,0.746201,0.737399,0.744554,0.752652,0.744573,0.747964,0.760611,0.753214,0.748002
max,0.999627,1.000000,0.998885,0.999858,0.999300,0.999617,0.999495,0.999997,0.998615,0.999924


TF4 y_train describe


count    3750.000000
mean        0.552895
std         0.263216
min         0.069815
25%         0.292075
50%         0.551363
75%         0.813203
max         0.999973
dtype: float64

TF4 y_test describe


count    1250.000000
mean        0.552624
std         0.262334
min         0.000000
25%         0.297225
50%         0.550549
75%         0.807680
max         1.000000
dtype: float64

In [31]:
# TF4 HKAN model params
tf4_model_params = [
    {
        "layer": 0,
        "n_vars_out": 1,
        "basis_fn": Tanh(s=3),
        "n_basis": 2,
        "centers": "equally_spaced",
        "expanding_base_regressor": Ridge(alpha=0.001),
    }
]
tf4_model = utils.build_hkan_model_from_configs(tf4_model_params)
tf4_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('expanding_layer_0', ...), ('connecting_layer_0', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_vars_out,1
,n_basis,2
,centers,'equally_spaced'
,basis_fn,<hkan.hkan.Ta...t 0x16eb90910>
,base_regressor,Ridge(alpha=0.001)
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True


In [32]:
# Fit HKAN for TF4
utils.fit_hkan(tf4_model, tf4_x_train, tf4_y_train)
tf4_results = em.evaluate(
    tf4_model,
    tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test
)

Fitting connecting regressors: 100%|██████████| 1/1 [00:00<00:00, 1122.37it/s]


✓ HKAN training completed in 0.0087 seconds


In [33]:
em.print_results(tf4_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             2.338550e-01         2.323804e-01        
RMSE            2.594233e-01         2.584713e-01        
-------------------------------------------------------
Inference (s)   0.001431             0.000511            


## KAN

In [34]:
tf4_kan_study = utils.study_optuna_kan("TF4", tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test, n_trials=10)

Results found for TF4. Loading from data/tf4_kan_optuna_search.csv
Best CV Validation RMSE: 0.263209
Best Test RMSE:          0.262005
Best width_idx:          1
Best architecture:       [10, 2, 1]

All 10 trials:
   number     value  params_width_idx  val_rmse  test_rmse
0       0  0.267264                 2  0.267264   0.264699
1       1  1.941438                 8  1.941438   0.578674
2       2  0.264042                 4  0.264042   0.262419
3       3  2.709694                 9  2.709694   0.873258
4       4  0.263209                 1  0.263209   0.262005
5       5  0.265406                 6  0.265406   0.262924
6       6  2.388389                 7  2.388389   0.611684
7       7  1.118305                 3  1.118305   1.554972
8       8  2.098387                 0  2.098387   2.034378
9       9  0.264162                 5  0.264162   0.267468


In [35]:
tf4_kan_model = utils.build_kan(width=tf4_kan_study["architecture"], grid=3, k=3, seed=42)

checkpoint directory created: ./model
saving model version 0.0


In [36]:
utils.fit_kan(tf4_kan_model, tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test, steps=100, opt="Adam", lamb=1e-3)

tf4_kan_results = em.evaluate(
    utils.KANModel(tf4_kan_model),
    tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test
)

| train_loss: 2.63e-01 | test_loss: 2.62e-01 | reg: 1.15e+01 | : 100%|█| 100/100 [00:06<00:00, 15.62

saving model version 0.1

✓ KAN training completed in 6.4077 seconds


In [37]:
em.print_results(tf4_kan_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             2.362001e-01         2.336923e-01        
RMSE            2.631507e-01         2.620049e-01        
-------------------------------------------------------
Inference (s)   0.010115             0.005096            


# TF5 Experiments

## HKAN

In [38]:
# Load TF5 dataset
tf5 = load_dataset("TF5")
tf5_x_train, tf5_x_test, tf5_y_train, tf5_y_test = tf5["train_input"], tf5["test_input"], tf5["train_label"], tf5["test_label"]

In [39]:
print("TF5 train describe")
display(pd.DataFrame(tf5_x_train).describe())
print("TF5 test describe")
display(pd.DataFrame(tf5_x_test).describe())
print("TF5 y_train describe")
display(pd.Series(tf5_y_train).describe())
print("TF5 y_test describe")
display(pd.Series(tf5_y_test).describe())

TF5 train describe


,0,1
count,5000.000000,5000.000000
mean,0.752401,0.750030
std,0.143031,0.142429
min,0.500216,0.500067
25%,0.630875,0.627742
50%,0.753696,0.749458
75%,0.874755,0.872681
max,0.999946,0.999991


TF5 test describe


,0,1
count,10000.000000,10000.000000
mean,0.750000,0.750000
std,0.145796,0.145796
min,0.500000,0.500000
25%,0.625000,0.625000
50%,0.750000,0.750000
75%,0.875000,0.875000
max,1.000000,1.000000


TF5 y_train describe


count    5000.000000
mean        0.884761
std         0.177206
min         0.007933
25%         0.818786
50%         0.991585
75%         0.999997
max         1.000000
dtype: float64

TF5 y_test describe


count    10000.000000
mean         0.884329
std          0.179202
min          0.000000
25%          0.818001
50%          0.992876
75%          0.999998
max          1.000000
dtype: float64

In [40]:
# tf5 HKAN model params
tf5_model_params = [
    {
        "layer": 0,
        "n_vars_out": 912,
        "basis_fn": Tanh(s=50),
        "n_basis": 23,
        "centers": "random",
        "expanding_base_regressor": Ridge(alpha=0.01),
    },
    {
        "layer": 1,
        "n_vars_out": 1,
        "basis_fn": Identity(),
        "n_basis": 1,
        "centers": "equally_spaced",
        "equally_spaced": None,
    }
]
tf5_model = utils.build_hkan_model_from_configs(tf5_model_params)
tf5_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('expanding_layer_0', ...), ('connecting_layer_0', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_vars_out,912
,n_basis,23
,centers,'random'
,basis_fn,<hkan.hkan.Ta...t 0x16ebc1350>
,base_regressor,Ridge(alpha=0.01)
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.01
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True


In [41]:
# Fit HKAN for TF5
utils.fit_hkan(tf5_model, tf5_x_train, tf5_y_train)
tf5_results = em.evaluate(
    tf5_model,
    tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test
)

Fitting connecting regressors: 100%|██████████| 1/1 [00:00<00:00,  4.45it/s]



✓ HKAN training completed in 6.8051 seconds


In [42]:
em.print_results(tf5_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             2.137590e-13         2.543793e-13        
RMSE            2.594402e-13         3.573463e-13        
-------------------------------------------------------
Inference (s)   2.725541             4.405965            


## KAN

In [43]:
tf5_kan_study = utils.study_optuna_kan("TF5", tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test, n_trials=10)

Results found for TF5. Loading from data/tf5_kan_optuna_search.csv
Best CV Validation RMSE: 0.177459
Best Test RMSE:          0.178417
Best width_idx:          0
Best architecture:       [2, 1]

All 10 trials:
   number     value  params_width_idx  val_rmse  test_rmse
0       0  0.196170                 2  0.196170   0.176963
1       1  0.336510                 8  0.336510   0.233948
2       2  0.537441                 4  0.537441   0.179316
3       3  0.673894                 9  0.673894   0.332759
4       4  0.186586                 1  0.186586   0.181430
5       5  0.347398                 6  0.347398   0.674302
6       6  0.382877                 7  0.382877   0.947079
7       7  0.183603                 3  0.183603   0.194581
8       8  0.177459                 0  0.177459   0.178417
9       9  0.644387                 5  0.644387   4.034507


In [44]:
tf5_kan_model = utils.build_kan(width=tf5_kan_study["architecture"], grid=3, k=3, seed=42)

checkpoint directory created: ./model
saving model version 0.0


In [45]:
utils.fit_kan(tf5_kan_model, tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test, steps=100, opt="Adam", lamb=1e-3)

tf5_kan_results = em.evaluate(
    utils.KANModel(tf5_kan_model),
    tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test
)

| train_loss: 1.77e-01 | test_loss: 1.79e-01 | reg: 4.63e-01 | : 100%|█| 100/100 [00:18<00:00,  5.30

saving model version 0.1

✓ KAN training completed in 18.8674 seconds


In [46]:
em.print_results(tf5_kan_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             1.415190e-01         1.427815e-01        
RMSE            1.765155e-01         1.784173e-01        
-------------------------------------------------------
Inference (s)   0.002277             0.002727            
